In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
base = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
for pat in ["fig_confound*", "fig_crossdataset*", "fig_fewshot*",
            "fig1_pipeline*", "d1_confusion*", "d4_roc*"]:
    hits = glob.glob(f"{base}/**/{pat}", recursive=True)
    print(pat, "->", hits if hits else "NOT FOUND")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ---- VERIFIED NUMBERS (from cross_dataset_confound.json + Tea_Leaf_Re_ cell 5) ----
# Left panel: mean leaf area (fraction of frame) by class
classes = ["Healthy", "Algal\nLeaf Spot", "Red\nSpider", "Gray\nBlight",
           "Helopeltis", "Brown\nBlight", "Red Rust"]
areas   = [0.267, 0.171, 0.165, 0.155, 0.117, 0.112, 0.109]

# Right panel: the three controls vs the headline
labels = ["Attention\npipeline\n(headline)", "Leaf geometry\nonly\n(C1)",
          "Scale-invariant\nreformulation\n(C3)", "Within-leaf\nattention\n(C2)"]
aucs   = [0.905, 0.834, 0.534, 0.543]
is_control = [False, True, True, True]

# ---- style ----
plt.rcParams.update({"font.size": 11, "font.family": "sans-serif",
                     "axes.spines.top": False, "axes.spines.right": False})
HEALTHY, DISEASE = "#2e7d32", "#c62828"
HEAD, CTRL, CHANCE = "#37474f", "#c62828", "#9e9e9e"

fig = plt.figure(figsize=(11, 4.2))
gs = GridSpec(1, 2, width_ratios=[1.05, 1], wspace=0.28)

# ===== LEFT: mean leaf area by class =====
ax1 = fig.add_subplot(gs[0])
colors = [HEALTHY] + [DISEASE] * (len(classes) - 1)
bars = ax1.bar(range(len(classes)), areas, color=colors, width=0.68,
               edgecolor="white", linewidth=0.5)
ax1.set_xticks(range(len(classes)))
ax1.set_xticklabels(classes, fontsize=8.5)
ax1.set_ylabel("Mean leaf area (fraction of frame)")
ax1.set_ylim(0, 0.30)
ax1.set_title("Healthy leaves are 1.94$\\times$ larger than diseased on average",
              fontsize=10.5, pad=8)
for b, v in zip(bars, areas):
    ax1.text(b.get_x() + b.get_width()/2, v + 0.006, f"{v:.3f}",
             ha="center", va="bottom", fontsize=8)
ax1.axhline(np.mean(areas[1:]), ls="--", lw=1, color="#555",
            label=f"diseased mean = {np.mean(areas[1:]):.3f}")
ax1.legend(frameon=False, fontsize=8.5, loc="upper right")
ax1.text(-0.09, 1.02, "a", transform=ax1.transAxes, fontweight="bold", fontsize=14)

# ===== RIGHT: the three controls =====
ax2 = fig.add_subplot(gs[1])
bar_colors = [CTRL if c else HEAD for c in is_control]
bars2 = ax2.bar(range(len(labels)), aucs, color=bar_colors, width=0.6,
                edgecolor="white", linewidth=0.5)
ax2.axhline(0.5, ls=":", lw=1.2, color=CHANCE)
ax2.text(len(labels) - 0.4, 0.505, "chance", color=CHANCE, fontsize=8.5, va="bottom")
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, fontsize=8.5)
ax2.set_ylabel("Healthy-vs-diseased AUC")
ax2.set_ylim(0.4, 1.0)
ax2.set_title("Controls collapse the apparent result", fontsize=10.5, pad=8)
for b, v in zip(bars2, aucs):
    ax2.text(b.get_x() + b.get_width()/2, v + 0.012, f"{v:.3f}",
             ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.text(-0.09, 1.02, "b", transform=ax2.transAxes, fontweight="bold", fontsize=14)

plt.savefig("fig_confound.pdf", bbox_inches="tight", dpi=300)
plt.savefig("fig_confound.png", bbox_inches="tight", dpi=200)  # for quick preview
print("saved fig_confound.pdf  and  fig_confound.png")
plt.show()

In [ ]:
# ============================================================
#  SETUP CELL  — run once per fresh Colab session
# ============================================================
import os, sys, json, numpy as np, torch, torch.nn as nn
from PIL import Image
import torchvision.transforms as T

# --- 1. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- 2. Paths (edit BASE only if your folder name differs) ---
BASE        = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
OUT         = BASE
RESULTS_DIR = os.path.join(BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, BASE)          # so `import leakage_audit` etc. work
print("BASE exists      :", os.path.isdir(BASE))
print("RESULTS_DIR      :", RESULTS_DIR)

# --- 3. Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device           :", device)

# --- 4. Load frozen DINOv1 ViT-S/16 with a CLS-attention hook ---
# Pulls the model from torch.hub (facebookresearch/dino). Needs internet (Colab has it).
dino_v1 = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
dino_v1 = dino_v1.to(device).eval()
for p in dino_v1.parameters():
    p.requires_grad = False

N_HEADS = 6
_dv1_store = {}

# Hook the last attention block to capture CLS->patch attention, shape [1, heads, 197, 197]
def _attn_hook(module, inp, out):
    # recompute attention from the block's qkv to get per-head CLS attention
    pass

# DINO's ViT exposes get_last_selfattention(); we use it directly instead of a hook.
def dino_cls_attention(img_tensor):
    """Returns CLS->patch attention as [N_HEADS, 196] (14x14 grid)."""
    with torch.no_grad():
        a = dino_v1.get_last_selfattention(img_tensor.to(device))  # [1, heads, 197, 197]
    a = a[0, :, 0, 1:]            # CLS row, drop CLS-to-CLS -> [heads, 196]
    return a.cpu().numpy()

# --- 5. Image transform used for DINO (matches your tf_dv1) ---
tf_dv1 = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- 6. Convenience wrapper matching your notebooks' _dv1_store pattern ---
def run_dino(img_pil):
    """Populates _dv1_store['cls_attn'] = [1, heads, 196] like your other cells expect."""
    x = tf_dv1(img_pil.convert('RGB')).unsqueeze(0)
    attn = dino_cls_attention(x)              # [heads, 196]
    _dv1_store['cls_attn'] = [attn]           # wrap to match your indexing [0]
    return attn

# quick sanity check
try:
    import glob
    sample = glob.glob(os.path.join(BASE, "**", "*.jpg"), recursive=True)
    if sample:
        a = run_dino(Image.open(sample[0]))
        print("DINO OK          : attention shape", a.shape, "(heads x patches)")
    else:
        print("DINO loaded, no sample .jpg found under BASE to test on.")
except Exception as e:
    print("DINO test skipped:", e)

print("\nSetup complete. Available: RESULTS_DIR, OUT, device, dino_v1, tf_dv1, "
      "run_dino(), dino_cls_attention(), _dv1_store, N_HEADS")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm opencv-python-headless

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (parallel, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src = _count_files(TEA_LEAF_ROOT)
n_dst = _count_files(LOCAL_DATA)

if n_dst < n_src:
    t0 = time.time()
    tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src, dst = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(dst):
                tasks.append((src, dst))
    print(f"Copying {len(tasks)} files with {N_WORKERS} workers...")
    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))
    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.2,0.2,0.2), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_test = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1,3,224,224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return (self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')),
                self.imgs[idx]['is_diseased'])

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
import sys, os
BASE = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"   # edit if your folder name differs
sys.path.insert(0, BASE)

# sanity check: are the three modules actually there?
for m in ["shape_confound.py", "feature_shape_control.py", "leakage_audit.py"]:
    print(m, "->", "found" if os.path.exists(os.path.join(BASE, m)) else "MISSING")

In [ ]:
import sys, os
BASE = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"   # your confirmed folder
if BASE not in sys.path:
    sys.path.insert(0, BASE)

import numpy as np, json, cv2, torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import shape_confound as sc
import feature_shape_control as fsc
import leakage_audit as la
print("imports OK")

In [ ]:
import inspect, leakage_audit as la
print(inspect.signature(la.run_audit))

In [ ]:
# ============================================================
#  SEGMENTATION STRESS TEST  (Reviewer Q7)
#  Perturbs the leaf mask before crop_to_leaf, re-measures probe AUC.
#  PREREQS (run earlier in this session):
#    1. your SETUP cell  -> builds all_images, dino_v1, tf_dv1, device, RESULTS_DIR
#    2. your AUDIT cell   -> report, uf, imgs = la.run_audit(...)
# ============================================================
import sys, os
BASE = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"     # edit if different
if BASE not in sys.path: sys.path.insert(0, BASE)

import numpy as np, json, cv2, torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import shape_confound as sc
import feature_shape_control as fsc
import leakage_audit as la

# --- dependency check: fail loudly with a clear message ---
assert 'uf' in dir() and 'imgs' in dir(), \
    "Run your AUDIT cell first (report, uf, imgs = la.run_audit(...)) to get uf, imgs."
for name in ['all_images', 'dino_v1', 'tf_dv1', 'device', 'RESULTS_DIR']:
    assert name in dir(), f"'{name}' missing — run your SETUP cell first."

SEED = 0
train_s, test_s = la.grouped_split(uf, imgs, seed=SEED*1000+42, test_frac=0.2)
print(f"split: {len(train_s)} train / {len(test_s)} test")

def perturb_mask(mask, kind, rng):
    m = mask.astype(np.uint8)
    if kind == "clean":  return mask
    if kind == "erode":  return cv2.erode(m, np.ones((11,11),np.uint8)).astype(bool)
    if kind == "dilate": return cv2.dilate(m, np.ones((11,11),np.uint8)).astype(bool)
    if kind == "shift":
        M = np.float32([[1,0,0.12*m.shape[1]],[0,1,0.12*m.shape[0]]])
        return cv2.warpAffine(m, M, (m.shape[1], m.shape[0])).astype(bool)
    if kind == "jitter":
        return perturb_mask(mask, rng.choice(["erode","dilate","shift"]), rng)

def feats_for(recs, kind, rng):
    F, y = [], []
    for r in recs:
        try: img = Image.open(r['path']).convert('RGB')
        except Exception: continue
        m = sc.leaf_mask(img)
        if m is None: continue
        m = perturb_mask(m, kind, rng)
        if m is None or m.sum() < 20: continue
        crop = fsc.crop_to_leaf(img, m)
        if crop is None: continue
        with torch.no_grad():
            f = dino_v1(tf_dv1(crop).unsqueeze(0).to(device)).cpu().numpy()[0]
        f /= (np.linalg.norm(f)+1e-9)
        F.append(f); y.append(r['is_diseased'])
    return np.array(F), np.array(y)

rng = np.random.default_rng(0)

# --- train probe on 70 clean labels (10/class), on CLEAN crops ---
by_cls = {}
for r in train_s: by_cls.setdefault(r['class'], []).append(r)
seed70 = []
for c, v in by_cls.items():
    idx = rng.choice(len(v), min(10, len(v)), replace=False)
    seed70 += [v[i] for i in idx]
print(f"seed set: {len(seed70)} labels across {len(by_cls)} classes")

Ftr, ytr = feats_for(seed70, "clean", rng)
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
probe.fit(Ftr, ytr)

# --- evaluate on test set under each mask perturbation ---
out = {}
for kind in ["clean","erode","dilate","shift","jitter"]:
    F, y = feats_for(test_s, kind, rng)
    p = probe.predict_proba(F)[:,1]
    out[kind] = {"auc": float(roc_auc_score(y, p)), "n": int(len(y))}
    print(f"{kind:8s}  AUC={out[kind]['auc']:.4f}  (n={out[kind]['n']})")

json.dump(out, open(os.path.join(RESULTS_DIR,"segmentation_stress.json"),"w"), indent=2)
print("\nsaved segmentation_stress.json")

In [ ]:
report, uf, imgs = la.run_audit(
    LOCAL_DATA, seed=42, test_frac=0.2,
    phash_thresh=0, dhash_thresh=0,      # exact canonical-hash joins only
    use_embeddings=True, emb_cos_thresh=0.97,
    out_json=os.path.join(RESULTS_DIR, "leakage_audit.json"),
)
print(len(imgs), "images; check the cluster count matches the paper (~137)")

re

In [ ]:
import os, leakage_audit as la

report, uf, imgs = la.run_audit(
    LOCAL_DATA,                       # /tmp/tea_leaf_local
    seed=42, test_frac=0.2,
    phash_thresh=0, dhash_thresh=0,   # exact canonical-hash joins (your paper's setting)
    use_embeddings=False,             # NO embedding pass -> gives 137 clusters
    out_json=os.path.join(RESULTS_DIR, "leakage_audit.json"),
)
print("uf + imgs ready")
# expect: 137 clusters, 365 images (2.8%), 66 cross-class, test=2616, leaked 2.91%

In [ ]:
# ============================================================
#  SEGMENTATION STRESS TEST  (Reviewer Q7)
# ============================================================
import sys, os
BASE = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
if BASE not in sys.path: sys.path.insert(0, BASE)

import numpy as np, json, cv2, torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import shape_confound as sc
import feature_shape_control as fsc
import leakage_audit as la

assert 'uf' in dir() and 'imgs' in dir(), "Run Cell 1 (audit) first."
for name in ['dino_v1','tf_dv1','device','RESULTS_DIR']:
    assert name in dir(), f"'{name}' missing — run your SETUP cell first."

SEED = 0
train_s, test_s = la.grouped_split(uf, imgs, seed=SEED*1000+42, test_frac=0.2)
print(f"grouped_split: train={len(train_s)}  test={len(test_s)}")
assert len(train_s) > len(test_s), \
    f"Split looks wrong (train={len(train_s)}, test={len(test_s)}). Re-run Cell 1 with the 137-cluster settings."

def perturb_mask(mask, kind, rng):
    m = mask.astype(np.uint8)
    if kind == "clean":  return mask
    if kind == "erode":  return cv2.erode(m, np.ones((11,11),np.uint8)).astype(bool)
    if kind == "dilate": return cv2.dilate(m, np.ones((11,11),np.uint8)).astype(bool)
    if kind == "shift":
        M = np.float32([[1,0,0.12*m.shape[1]],[0,1,0.12*m.shape[0]]])
        return cv2.warpAffine(m, M, (m.shape[1], m.shape[0])).astype(bool)
    if kind == "jitter":
        return perturb_mask(mask, rng.choice(["erode","dilate","shift"]), rng)

def feats_for(recs, kind, rng):
    F, y = [], []
    for r in recs:
        try: img = Image.open(r['path']).convert('RGB')
        except Exception: continue
        m = sc.leaf_mask(img)
        if m is None: continue
        m = perturb_mask(m, kind, rng)
        if m is None or m.sum() < 20: continue
        crop = fsc.crop_to_leaf(img, m)
        if crop is None: continue
        with torch.no_grad():
            f = dino_v1(tf_dv1(crop).unsqueeze(0).to(device)).cpu().numpy()[0]
        f /= (np.linalg.norm(f)+1e-9)
        F.append(f); y.append(r['is_diseased'])
    return np.array(F), np.array(y)

rng = np.random.default_rng(0)

by_cls = {}
for r in train_s: by_cls.setdefault(r['class'], []).append(r)
seed70 = []
for c, v in by_cls.items():
    idx = rng.choice(len(v), min(10, len(v)), replace=False)
    seed70 += [v[i] for i in idx]
print(f"seed set: {len(seed70)} labels across {len(by_cls)} classes")

Ftr, ytr = feats_for(seed70, "clean", rng)
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
probe.fit(Ftr, ytr)

out = {}
for kind in ["clean","erode","dilate","shift","jitter"]:
    F, y = feats_for(test_s, kind, rng)
    out[kind] = {"auc": float(roc_auc_score(y, p:=probe.predict_proba(F)[:,1])), "n": int(len(y))}
    print(f"{kind:8s}  AUC={out[kind]['auc']:.4f}  (n={out[kind]['n']})")

json.dump(out, open(os.path.join(RESULTS_DIR,"segmentation_stress.json"),"w"), indent=2)
print("\nsaved segmentation_stress.json")

In [ ]:
# ============================================================
#  SSL BASELINE (Q4) — crop-cached, fast, no re-segmentation
# ============================================================
import sys, os, time
BASE="/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
if BASE not in sys.path: sys.path.insert(0,BASE)
import numpy as np, json, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import roc_auc_score
import shape_confound as sc, feature_shape_control as fsc, leakage_audit as la

assert 'uf' in dir() and 'imgs' in dir(), "Run the 137-cluster AUDIT cell first."
CACHE="/tmp/ssl_crops"; os.makedirs(CACHE, exist_ok=True)

# ---- 1. cache every crop ONCE as a 224x224 .npy (uint8) ----
def crop_path(rec):
    return os.path.join(CACHE, str(abs(hash(rec['path'])))+".npy")

def cache_crops(recs):
    made=0
    for r in recs:
        cp=crop_path(r)
        if os.path.exists(cp): continue
        try: img=Image.open(r['path']).convert('RGB')
        except: continue
        m=sc.leaf_mask(img); c=fsc.crop_to_leaf(img,m) if m is not None else None
        if c is None: c=img.resize((224,224))
        np.save(cp, np.asarray(c.resize((224,224)), dtype=np.uint8)); made+=1
    return made

# cache the union of everything we'll touch (train + test), once
train0, test0 = la.grouped_split(uf, imgs, seed=42, test_frac=0.2)
t=time.time(); n=cache_crops(train0+test0)
print(f"cached {n} new crops ({len(os.listdir(CACHE))} total) in {time.time()-t:.0f}s")

# ---- 2. dataset reads cached crops (fast, no segmentation) ----
NORM=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf_tr=T.Compose([T.ToPILImage(),T.RandomHorizontalFlip(),T.RandomVerticalFlip(),
                 T.RandomRotation(20),T.ColorJitter(0.2,0.2,0.2),T.ToTensor(),NORM])
tf_ev=T.Compose([T.ToPILImage(),T.ToTensor(),NORM])

class CropDS(Dataset):
    def __init__(s,recs,labels,tf): s.recs,s.labels,s.tf=recs,labels,tf
    def __len__(s): return len(s.recs)
    def __getitem__(s,i):
        arr=np.load(crop_path(s.recs[i]))          # cached uint8 224x224x3
        return s.tf(arr), int(s.labels[i])

def new_student():
    m=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,2); return m.to(device)

def train(model,ds,epochs=30,lr=1e-3):
    dl=DataLoader(ds,batch_size=64,shuffle=True,num_workers=0)   # workers=0 avoids the hang
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs); lf=nn.CrossEntropyLoss()
    model.train()
    for ep in range(epochs):
        for x,y in dl:
            x,y=x.to(device),y.to(device); opt.zero_grad()
            lf(model(x),y).backward(); opt.step()
        sch.step()
    return model

@torch.no_grad()
def predict(model,recs):
    model.eval(); dl=DataLoader(CropDS(recs,[0]*len(recs),tf_ev),batch_size=256,num_workers=0)
    return np.concatenate([torch.softmax(model(x.to(device)),1)[:,1].cpu().numpy() for x,_ in dl])

# ---- 3. run (1 seed first; set SEEDS=range(3) later for error bars) ----
SEEDS=range(1)
results={}
for SEED in SEEDS:
    rng=np.random.default_rng(SEED)
    tr,te=la.grouped_split(uf,imgs,seed=SEED*1000+42,test_frac=0.2)
    cache_crops(tr+te)                              # ensure this split's crops exist
    yte=np.array([r['is_diseased'] for r in te])
    byc={};  [byc.setdefault(r['class'],[]).append(r) for r in tr]
    s70,s70y=[],[]
    for c,v in byc.items():
        for i in rng.choice(len(v),min(10,len(v)),replace=False): s70.append(v[i]); s70y.append(v[i]['is_diseased'])
    pool=[r for r in tr if r not in s70]
    m0=train(new_student(),CropDS(s70,s70y,tf_tr))
    auc_sup=roc_auc_score(yte,predict(m0,te))
    pl=(predict(m0,pool)>0.5).astype(int)
    m1=train(new_student(),CropDS(s70+pool,list(s70y)+list(pl),tf_tr))
    auc_self=roc_auc_score(yte,predict(m1,te))
    results[SEED]={"supervised_70":float(auc_sup),"selftrain_70":float(auc_self)}
    print(f"seed {SEED}: supervised={auc_sup:.4f}  self-train={auc_self:.4f}")

json.dump(results,open(os.path.join(RESULTS_DIR,"ssl_baseline.json"),"w"),indent=2)
print("compare vs your probe+distillation student: 0.973")
print("saved ssl_baseline.json")

In [ ]:
# ============================================================
#  Q4 COMPLETE: SSL baseline (3 seeds) + encoder note
#  Crops already cached in /tmp/ssl_crops from the prior run.
# ============================================================
import sys, os, time
BASE="/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
if BASE not in sys.path: sys.path.insert(0,BASE)
import numpy as np, json, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import roc_auc_score, accuracy_score
import shape_confound as sc, feature_shape_control as fsc, leakage_audit as la

assert 'uf' in dir() and 'imgs' in dir(), "Run the 137-cluster AUDIT cell first."
CACHE="/tmp/ssl_crops"; os.makedirs(CACHE, exist_ok=True)
def crop_path(rec): return os.path.join(CACHE, str(abs(hash(rec['path'])))+".npy")

def cache_crops(recs):
    made=0
    for r in recs:
        cp=crop_path(r)
        if os.path.exists(cp): continue
        try: img=Image.open(r['path']).convert('RGB')
        except: continue
        m=sc.leaf_mask(img); c=fsc.crop_to_leaf(img,m) if m is not None else None
        if c is None: c=img.resize((224,224))
        np.save(cp, np.asarray(c.resize((224,224)),dtype=np.uint8)); made+=1
    return made

NORM=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf_tr=T.Compose([T.ToPILImage(),T.RandomHorizontalFlip(),T.RandomVerticalFlip(),
                 T.RandomRotation(20),T.ColorJitter(0.2,0.2,0.2),T.ToTensor(),NORM])
tf_ev=T.Compose([T.ToPILImage(),T.ToTensor(),NORM])

class CropDS(Dataset):
    def __init__(s,recs,labels,tf): s.recs,s.labels,s.tf=recs,labels,tf
    def __len__(s): return len(s.recs)
    def __getitem__(s,i):
        return s.tf(np.load(crop_path(s.recs[i]))), int(s.labels[i])

def new_student():
    m=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,2); return m.to(device)

def train(model,ds,epochs=30,lr=1e-3):
    dl=DataLoader(ds,batch_size=64,shuffle=True,num_workers=0)
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs); lf=nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for x,y in dl:
            x,y=x.to(device),y.to(device); opt.zero_grad(); lf(model(x),y).backward(); opt.step()
        sch.step()
    return model

@torch.no_grad()
def predict(model,recs):
    model.eval(); dl=DataLoader(CropDS(recs,[0]*len(recs),tf_ev),batch_size=256,num_workers=0)
    return np.concatenate([torch.softmax(model(x.to(device)),1)[:,1].cpu().numpy() for x,_ in dl])

results={}
for SEED in range(3):
    t0=time.time()
    rng=np.random.default_rng(SEED)
    tr,te=la.grouped_split(uf,imgs,seed=SEED*1000+42,test_frac=0.2)
    cache_crops(tr+te)
    yte=np.array([r['is_diseased'] for r in te])
    byc={};  [byc.setdefault(r['class'],[]).append(r) for r in tr]
    s70,s70y=[],[]
    for c,v in byc.items():
        for i in rng.choice(len(v),min(10,len(v)),replace=False): s70.append(v[i]); s70y.append(v[i]['is_diseased'])
    pool=[r for r in tr if r not in s70]
    # supervised-only
    m0=train(new_student(),CropDS(s70,s70y,tf_tr))
    ps=predict(m0,te); auc_sup=roc_auc_score(yte,ps); acc_sup=accuracy_score(yte,ps>0.5)
    # self-training
    pl=(predict(m0,pool)>0.5).astype(int)
    m1=train(new_student(),CropDS(s70+pool,list(s70y)+list(pl),tf_tr))
    pf=predict(m1,te); auc_self=roc_auc_score(yte,pf); acc_self=accuracy_score(yte,pf>0.5)
    results[SEED]={"supervised_auc":float(auc_sup),"supervised_acc":float(acc_sup),
                   "selftrain_auc":float(auc_self),"selftrain_acc":float(acc_self)}
    print(f"seed {SEED}: sup AUC={auc_sup:.4f} acc={acc_sup:.3f} | self AUC={auc_self:.4f} acc={acc_self:.3f}  ({time.time()-t0:.0f}s)")

sa=np.array([results[s]['supervised_auc'] for s in results])
fa=np.array([results[s]['selftrain_auc'] for s in results])
summary={"supervised_only":{"auc_mean":float(sa.mean()),"auc_std":float(sa.std())},
         "selftraining":{"auc_mean":float(fa.mean()),"auc_std":float(fa.std())},
         "probe_distillation_reference":0.973,
         "encoder_ablation":{"DINOv1_ViTS16":0.896,"DINOv2_ViTS14":0.823,"CLIP_ViTB16":0.704},
         "per_seed":results,"n_seeds":3,"budget_labels":70}
json.dump(summary,open(os.path.join(RESULTS_DIR,"ssl_baseline.json"),"w"),indent=2)
print(f"\nsupervised-only : {sa.mean():.4f} +/- {sa.std():.4f}")
print(f"self-training   : {fa.mean():.4f} +/- {fa.std():.4f}")
print(f"your pipeline   : 0.973 (probe + distillation)")
print("saved ssl_baseline.json")

In [ ]:
import json, os
print(open(os.path.join(RESULTS_DIR, "phase3_model_comparison.json")).read())